# Math Photo Solver — Training on Google Colab

Цей ноутбук виконує повний цикл:
1. Клонує репозиторій
2. Встановлює залежності
3. Генерує датасет символів (80/20)
4. Навчає ResNet-18 класифікатор символів
5. Оцінює модель
6. Зберігає ваги на Google Drive

> **Перед запуском**: `Среда выполнения → Сменить тип среды выполнения → GPU (T4)`

## Крок 1. Підключення Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_SAVE_DIR = '/content/drive/MyDrive/math_solver_models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print('Drive підключено. Модель буде збережена в:', DRIVE_SAVE_DIR)

## Крок 2. Клонування репозиторію

In [ ]:
# Замінити URL на URL свого репозиторію
REPO_URL = 'https://github.com/sergey2321/sergey2321.git'

!git clone {REPO_URL} /content/math-solver
%cd /content/math-solver
!git checkout claude/ale-yP87K
print('Репозиторій клоновано.')

## Крок 3. Встановлення залежностей

In [ ]:
!pip install -q -r requirements.txt
print('Залежності встановлено.')

## Крок 4. Перевірка GPU

In [ ]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('УВАГА: GPU не знайдено. Навчання буде повільним.')

## Крок 5. Генерація датасету символів (80/20)

In [ ]:
# Кількість зображень — збільш для кращої точності (рекомендовано 20000+)
DATASET_COUNT = 10000
DATASET_DIR   = 'dataset/symbols'

!python -m dataset.generator.generate_handwritten \
    --count {DATASET_COUNT} \
    --out {DATASET_DIR} \
    --seed 42

import os
train_count = len(os.listdir(f'{DATASET_DIR}/train'))
val_count   = len(os.listdir(f'{DATASET_DIR}/val'))
print(f'Train: {train_count}  |  Val: {val_count}')

## Крок 6. Навчання моделі

In [ ]:
EPOCHS     = 25
BATCH_SIZE = 128
LR         = 1e-3
MODEL_OUT  = 'backend/models/symbol_clf.pth'

!python -m training.train_ocr \
    --data {DATASET_DIR} \
    --epochs {EPOCHS} \
    --batch {BATCH_SIZE} \
    --lr {LR} \
    --out {MODEL_OUT}

## Крок 7. Оцінка моделі

In [ ]:
!python -m training.evaluate \
    --data {DATASET_DIR} \
    --model {MODEL_OUT}

## Крок 8. Збереження моделі на Google Drive

In [ ]:
import shutil
from datetime import datetime

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
dest = f'{DRIVE_SAVE_DIR}/symbol_clf_{timestamp}.pth'
shutil.copy(MODEL_OUT, dest)
print(f'Модель збережена на Drive: {dest}')

# Також зберігаємо під стандартним ім'ям
shutil.copy(MODEL_OUT, f'{DRIVE_SAVE_DIR}/symbol_clf_latest.pth')
print(f'Також збережена як: {DRIVE_SAVE_DIR}/symbol_clf_latest.pth')

## Крок 9. Завантаження моделі на локальний комп'ютер

Після завершення навчання:

**Спосіб 1** — Завантажити прямо з Colab:
```python
from google.colab import files
files.download('backend/models/symbol_clf.pth')
```

**Спосіб 2** — Взяти з Google Drive (файл уже там):
`/MyDrive/math_solver_models/symbol_clf_latest.pth`

Скопіюй файл у `backend/models/symbol_clf.pth` у локальному репозиторії.

In [ ]:
# Розкоментуй для прямого завантаження
# from google.colab import files
# files.download('backend/models/symbol_clf.pth')

## Додатково: Візуалізація датасету

Покажемо кілька прикладів згенерованих символів.

In [ ]:
import json
import matplotlib.pyplot as plt
from PIL import Image
import random

with open(f'{DATASET_DIR}/metadata.json') as f:
    meta = json.load(f)

samples = random.sample(meta['train'], min(20, len(meta['train'])))

fig, axes = plt.subplots(2, 10, figsize=(20, 5))
for ax, rec in zip(axes.flat, samples):
    img = Image.open(f"{DATASET_DIR}/{rec['file']}")
    ax.imshow(img, cmap='gray')
    ax.set_title(rec['symbol'], fontsize=10)
    ax.axis('off')

plt.suptitle('Приклади символів із датасету', fontsize=14)
plt.tight_layout()
plt.show()